# M2/F6 in SageMaker Studio JupyterLab

Prepared, NOT RUN. Use the built-in SageMaker Distribution GPU image and **ml.g6.2xlarge**.
No custom Docker image, ECR, CodeBuild or managed training job is used by this notebook.

**After F6: verify S3 backup, STOP/DELETE this JupyterLab app, and verify Deleted. Closing the
browser does not stop GPU billing. Delete the space/EBS only after the results are safely backed up.**


## 1. Load the prepared handoff
Before starting compute, upload the input ZIP to the exact S3 key documented in the handoff.
Upload this notebook, `launch_spec.json`, `infra.json`, and the entire `control/` folder to the
same Studio directory. Use the Studio execution role; do not paste credentials or mount Drive.


In [ ]:
import json
import sys
from pathlib import Path

HANDOFF = Path.cwd()
sys.path.insert(0, str(HANDOFF / 'control'))
from apexpinn.cloud import studio_setup, studio_runner

spec = json.loads((HANDOFF / 'launch_spec.json').read_text())
infra = json.loads((HANDOFF / 'infra.json').read_text())
HOME_ROOT = Path.home() / 'apexpinn'
RUN_ID = 'm2-f6-mid-studio-v1'
STATE = HOME_ROOT / 'runs' / RUN_ID
print({'selection': spec['selection'], 'instance_type': infra['instance_type'], 'status': 'NOT RUN'})
print(studio_runner.REMINDER)


## 2. Verify and restore the input ZIP
File transfer and SHA verification only. The workspace and run records live on the persistent
Studio space volume. Keep RUN_ID, input/source and environment unchanged when continuing a run.


In [ ]:
WORKSPACE = studio_setup.prepare_workspace(spec, infra, HOME_ROOT)
print('Verified workspace:', WORKSPACE)


## 3. Prepare Python 3.12 and verify the L4 GPU
Run this remotely in Studio. uv installs the frozen CUDA environment under WORKSPACE/.venv.
The notebook kernel can use a different Python version. No fitting happens in this cell.


In [ ]:
PYTHON = studio_setup.install_environment(WORKSPACE)
print('Scientific Python:', PYTHON)


## 4. Preview the exact F6 command
The numerical protocol is unchanged:20 tuning trials,5 calibration fits,5 final fits and the
existing horizons/evaluation. A small remote checkpoint restart oracle runs first, separately
from those fit slots. Existing F1-F5 results are not replayed. This cell does not train.


In [ ]:
import subprocess
command = [str(PYTHON), 'scripts/run_models.py', '--method', 'configs/t13_studio_v1.yaml',
           '--out', str(STATE / 'run'), '--model', 'M2', '--fold', 'F6',
           '--phi', 'mid', '--variant', 'main']
subprocess.run(command + ['--plan'], cwd=WORKSPACE, check=True)


## 5. User-launched training
Set RUN_TRAINING=True only when ready. Keep this foreground cell running so its kernel remains
busy; browser disconnection does not imply server termination. Studio idle shutdown is a separate
policy and must be verified during the remote run. The 24h process timeout does NOT stop GPU billing.

Full-state checkpoints and attempt logs are saved on the space volume. A verified immutable S3
snapshot is attempted every5 minutes and at exit. Sync delay or an abrupt loss can lose work since
the last committed snapshot. Resume restores only compatible verified state, never failed fit slots.


In [ ]:
RUN_TRAINING = False
try:
    if RUN_TRAINING:
        studio_runner.run(PYTHON, WORKSPACE, STATE, bucket=infra['bucket_name'],
                          prefix='studio-runs/' + RUN_ID, bundle_sha=spec['bundle_sha256'])
    else:
        print('NOT RUN. GPU compute remains billable while the Studio app is running.')
finally:
    print(studio_runner.REMINDER)


## 6. Review logs and completion
Process .log files and JSON outcome records are in STATE/logs. Inspect progress.json timestamps
and step movement, not the notebook timer. After a disconnect, inspect those records before
rerunning the training cell; the local lock prevents concurrent invocation on this app.


In [ ]:
print(json.dumps(studio_setup.progress(STATE), indent=2))
print('Persistent run folder:', STATE)
print('S3 snapshots:', 's3://' + infra['bucket_name'] + '/studio-runs/' + RUN_ID + '/')


## 7. Completion reminder: stop and delete compute

1. Require exit0, job.json, report and verified exported results before declaring F6 complete.
2. Verify the final S3 snapshot and download it for local artifact review.
3. In Studio, **Stop the JupyterLab app**. AWS DeleteApp stops/removes that app's compute instance.
4. Verify app status **Deleted**. Closing this tab/browser is insufficient.
5. Delete the private space only after S3 backup verification if you also want to remove its EBS
   storage. The IaC protects the space and S3 against accidental destruction. Domain EFS may remain;
   review retained storage separately. Preserve the scientific S3 archives.

Do not delete the space while training, backup or verification is incomplete. No automatic app or
space deletion is performed by this notebook. I will repeat this checklist when reviewing F6 results.


In [ ]:
print(studio_runner.REMINDER)
